In [1]:
import redis
import time
import base64
import json
import csv
import random
import statistics

import pandas as pd

In [2]:
def sliding_window(data, window_size, step):
    for start_row in range(0, len(data) - window_size + 1, step):
        yield data[start_row:start_row + window_size]        

In [3]:
#test = pd.read_csv("text_1.csv", header=None)
#test = pd.read_csv("text_2.csv", header=None)
test = pd.read_csv("text_3.csv", header=None)

In [4]:
test = test.transpose()
test

,0
0,"b'{""ReceivedTopic"":"""",""CorrelationID"":"""",""Payl..."
1,"b'{""ReceivedTopic"":"""",""CorrelationID"":"""",""Payl..."
2,"b'{""ReceivedTopic"":"""",""CorrelationID"":"""",""Payl..."
3,"b'{""ReceivedTopic"":"""",""CorrelationID"":"""",""Payl..."
4,"b'{""ReceivedTopic"":"""",""CorrelationID"":"""",""Payl..."
...,...
995,"b'{""ReceivedTopic"":"""",""CorrelationID"":"""",""Payl..."
996,"b'{""ReceivedTopic"":"""",""CorrelationID"":"""",""Payl..."
997,"b'{""ReceivedTopic"":"""",""CorrelationID"":"""",""Payl..."
998,"b'{""ReceivedTopic"":"""",""CorrelationID"":"""",""Payl..."


In [5]:
humid = []
pm10 = []
pm25 = []
temp = []

for i in range(len(test)):
    val = test.values[i][0]
    val = val.replace("'", "")
    val = val.replace("b","",1)
    json_val = json.loads(val)
    
    res_payload = json_val['Payload']
    dec_res = base64.b64decode(res_payload)
    dec_res = dec_res.decode("UTF-8")
    str_test = dec_res.replace("'","\"")
    json_data = json.loads(str_test)
    
    humid.append(json_data['event']['readings'][0]['objectValue']['humidity'])
    pm10.append(json_data['event']['readings'][0]['objectValue']['pm10'])
    pm25.append(json_data['event']['readings'][0]['objectValue']['pm25'])
    temp.append(json_data['event']['readings'][0]['objectValue']['temperature'])

In [6]:
#차분 양수/음수 분리
diff_humid_p = []
diff_pm10_p = []
diff_pm25_p = []
diff_temp_p = []

diff_humid_n = []
diff_pm10_n = []
diff_pm25_n = []
diff_temp_n = []

for i in range(len(humid)-1):    
    diff_humid = humid[i+1]-humid[i]
    diff_pm10 = pm10[i+1]-pm10[i]
    diff_pm25 = pm25[i+1]-pm25[i]
    diff_temp = temp[i+1]-temp[i]
    
    if diff_humid < 0 :
        diff_humid_n.append(diff_humid)
    else:
        diff_humid_p.append(diff_humid)
        
    if diff_pm10 < 0 :
        diff_pm10_n.append(diff_humid)
    else:
        diff_pm10_p.append(diff_humid)

    if diff_pm25 < 0 :
        diff_pm25_n.append(diff_humid)
    else:
        diff_pm25_p.append(diff_humid)
        
    if diff_temp < 0 :
        diff_temp_n.append(diff_humid)
    else:
        diff_temp_p.append(diff_humid)   

In [7]:
humid_window_p = list(sliding_window(diff_humid_p,5,100))
pm10_window_p = list(sliding_window(diff_pm10_p,5,100))
pm25_window_p = list(sliding_window(diff_pm25_p,5,100))
temp_window_p = list(sliding_window(diff_temp_p,5,100))

humid_window_n = list(sliding_window(diff_humid_n,5,100))
pm10_window_n = list(sliding_window(diff_pm10_n,5,100))
pm25_window_n = list(sliding_window(diff_pm25_n,5,100))
temp_window_n = list(sliding_window(diff_temp_n,5,100))

In [8]:
avg_dif_humid_p = []
avg_dif_pm10_p = []
avg_dif_pm25_p = []
avg_dif_temp_p = []

avg_dif_humid_n = []
avg_dif_pm10_n = []
avg_dif_pm25_n = []
avg_dif_temp_n = []

for i in range(len(humid_window_p)):
    avg_dif_humid_p.append(statistics.mean(humid_window_p[i]))

for i in range(len(pm10_window_p)):
    avg_dif_pm10_p.append(statistics.mean(pm10_window_p[i]))

for i in range(len(pm25_window_p)):
    avg_dif_pm25_p.append(statistics.mean(pm25_window_p[i]))
    
for i in range(len(temp_window_p)):
    avg_dif_temp_p.append(statistics.mean(temp_window_p[i]))
    
for i in range(len(humid_window_n)):
    avg_dif_humid_n.append(statistics.mean(humid_window_n[i]))

for i in range(len(pm10_window_n)):
    avg_dif_pm10_n.append(statistics.mean(pm10_window_n[i]))

for i in range(len(pm25_window_n)):
    avg_dif_pm25_n.append(statistics.mean(pm25_window_n[i]))
    
for i in range(len(temp_window_n)):
    avg_dif_temp_n.append(statistics.mean(temp_window_n[i]))

In [9]:
total_avdf_humid_p = statistics.mean(avg_dif_humid_p)
total_avdf_pm10_p = statistics.mean(avg_dif_pm10_p)
total_avdf_pm25_p = statistics.mean(avg_dif_pm25_p)
total_avdf_temp_p = statistics.mean(avg_dif_temp_p)

total_avdf_humid_n = statistics.mean(avg_dif_humid_n)
total_avdf_pm10_n = statistics.mean(avg_dif_pm10_n)
total_avdf_pm25_n = statistics.mean(avg_dif_pm25_n)
total_avdf_temp_n = statistics.mean(avg_dif_temp_n)

In [10]:
humid_del_list = []
pm10_del_list = []
pm25_del_list = []
temp_del_list = []

for i in range(len(humid)-1):    
    diff_humid = humid[i+1]-humid[i]
    
    if diff_humid < 0:
        if abs(diff_humid) < abs(total_avdf_humid_n):
            humid_del_list.append(i)
    else:
        if abs(diff_humid) < abs(total_avdf_humid_p):
            humid_del_list.append(i)

for i in range(len(pm10)-1):
    diff_pm10 = pm10[i+1]-pm10[i]
    
    if diff_pm10 < 0:
        if abs(diff_pm10) < abs(total_avdf_pm10_n):
            pm10_del_list.append(i)
    else:
        if abs(diff_pm10) < abs(total_avdf_pm10_p):
            pm10_del_list.append(i)
            
for i in range(len(pm25)-1):    
    diff_pm25 = pm25[i+1]-pm25[i]
    
    if diff_pm25 < 0:
        if abs(diff_pm25) < abs(total_avdf_pm25_n):
            pm25_del_list.append(i)
    else:
        if abs(diff_pm25) < abs(total_avdf_pm25_p):
            pm25_del_list.append(i)
            
for i in range(len(temp)-1):    
    diff_temp = temp[i+1]-temp[i]
    
    if diff_temp < 0:
        if abs(diff_temp) < abs(total_avdf_temp_n):
            temp_del_list.append(i)
    else:
        if abs(diff_temp) < abs(total_avdf_temp_p):
            temp_del_list.append(i)

In [11]:
len(humid_del_list)

655

In [12]:
len(pm10_del_list)

201

In [13]:
len(pm25_del_list)

186

In [14]:
len(temp_del_list)

275

In [15]:
step1 = list(set(humid_del_list).intersection(pm10_del_list))
step2 = list(set(step1).intersection(pm25_del_list))
step3 = list(set(step2).intersection(temp_del_list))

In [19]:
len(step3)

7